# Step03 no-site-filter all135k MPS run

这个 notebook 读取本文件夹里的 `step02_input`，用于复跑 135,300 个 m6A site 的 stress test。深度学习训练强制使用 Apple Silicon `mps`，如果 MPS 不可用会直接停止，不会回退 CPU。

说明：CSV/gzip 读取、DataFrame 整理、Matplotlib 画图会使用 CPU，这是文件 I/O 和可视化，不是深度学习训练。模型 forward/backward/loss、site_mean baseline、PCA 矩阵乘法、k-means 距离计算都放在 MPS tensor 上。

In [ ]:
from __future__ import annotations

import gc
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR = Path('/Users/Zhuanz/Desktop/FYP/SigRM-master/data/FYPmain/no_site_filter_runs/20260414_all135k')
INPUT_DIR = BASE_DIR / 'step02_input'
OUTPUT_DIR = BASE_DIR / 'step03_notebook_mps_pca_cluster_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
LATENT_DIM = 8
HIDDEN_DIM_1 = 256
HIDDEN_DIM_2 = 64
DROPOUT = 0.05
INPUT_NOISE = 0.05
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 64
MAX_EPOCHS = 30
MIN_EPOCHS = 5
PATIENCE = 5
TOTAL_CAP = 200.0
SELECTION_SIGNIFICANT_WEIGHT = 0.70
NEIGHBOR_K = 15
PCA_COMPONENTS = 8
PCA_OVERSAMPLE = 16
PREDICTION_SCATTER_MAX_POINTS = 50000

if not torch.backends.mps.is_available():
    raise RuntimeError('MPS is not available. Refusing to run deep learning on CPU.')
DEVICE = torch.device('mps')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print({'device': str(DEVICE), 'torch': torch.__version__, 'input_dir': str(INPUT_DIR), 'output_dir': str(OUTPUT_DIR)})


In [ ]:
def load_matrix(path: Path, dtype=np.float32):
    print('loading', path.name, flush=True)
    df = pd.read_csv(path)
    row_ids = df.iloc[:, 0].astype(str).to_numpy()
    col_ids = df.columns[1:].astype(str).to_numpy()
    arr = df.iloc[:, 1:].to_numpy(dtype=dtype, copy=True)
    print('loaded', path.name, arr.shape, flush=True)
    return row_ids, col_ids, arr

def to_mps(arr: np.ndarray) -> torch.Tensor:
    return torch.tensor(arr, dtype=torch.float32, device=DEVICE)

def make_input_features(ratio_t: torch.Tensor, mask_t: torch.Tensor, noise_rate: float = 0.0) -> torch.Tensor:
    if noise_rate > 0:
        dropped = (torch.rand_like(ratio_t) < noise_rate) & (mask_t > 0)
        ratio_t = ratio_t.masked_fill(dropped, 0.0)
        mask_t = mask_t.masked_fill(dropped, 0.0)
    return torch.cat([ratio_t * mask_t, mask_t], dim=1)

class BinomialAutoencoder(nn.Module):
    def __init__(self, n_sites: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_sites * 2, HIDDEN_DIM_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM_1, HIDDEN_DIM_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM_2, LATENT_DIM),
        )
        self.decoder = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_DIM_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM_2, HIDDEN_DIM_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM_1, n_sites), nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        return self.decoder(z), z

def masked_binomial_nll(pred: torch.Tensor, m6a_t: torch.Tensor, total_t: torch.Tensor, mask_t: torch.Tensor) -> torch.Tensor:
    p = torch.clamp(pred, 1e-6, 1 - 1e-6)
    effective_total = torch.clamp(total_t, max=TOTAL_CAP)
    weights = effective_total / torch.clamp(total_t, min=1.0)
    nll = -(m6a_t * torch.log(p) + (total_t - m6a_t) * torch.log1p(-p))
    denom = torch.clamp((effective_total * mask_t).sum(), min=1.0)
    return (nll * weights * mask_t).sum() / denom

def masked_ratio_mae(pred: torch.Tensor, ratio_t: torch.Tensor, mask_t: torch.Tensor) -> torch.Tensor:
    return ((pred - ratio_t).abs() * mask_t).sum() / torch.clamp(mask_t.sum(), min=1.0)

def site_mean_prediction(ratio_t: torch.Tensor, mask_t: torch.Tensor) -> torch.Tensor:
    return (ratio_t * mask_t).sum(dim=0) / torch.clamp(mask_t.sum(dim=0), min=1.0)

def evaluate_prediction_vector(site_mean_t: torch.Tensor, mask_t: torch.Tensor, ratio_t: torch.Tensor, m6a_t: torch.Tensor, total_t: torch.Tensor):
    pred = site_mean_t.unsqueeze(0).expand_as(ratio_t)
    return float(masked_binomial_nll(pred, m6a_t, total_t, mask_t).cpu()), float(masked_ratio_mae(pred, ratio_t, mask_t).cpu())

def evaluate_model(model: nn.Module, input_mask_t: torch.Tensor, eval_mask_t: torch.Tensor, ratio_t: torch.Tensor, m6a_t: torch.Tensor, total_t: torch.Tensor, batch_size: int = 32):
    model.eval()
    nll_num = 0.0
    nll_den = 0.0
    mae_num = 0.0
    mae_den = 0.0
    with torch.inference_mode():
        for start in range(0, ratio_t.shape[0], batch_size):
            sl = slice(start, min(start + batch_size, ratio_t.shape[0]))
            br, bm6a, bt = ratio_t[sl], m6a_t[sl], total_t[sl]
            binput_mask, beval_mask = input_mask_t[sl], eval_mask_t[sl]
            pred, _ = model(make_input_features(br, binput_mask, 0.0))
            p = torch.clamp(pred, 1e-6, 1 - 1e-6)
            effective_total = torch.clamp(bt, max=TOTAL_CAP)
            weights = effective_total / torch.clamp(bt, min=1.0)
            nll = -(bm6a * torch.log(p) + (bt - bm6a) * torch.log1p(-p)) * weights
            nll_num += float((nll * beval_mask).sum().cpu())
            nll_den += float((effective_total * beval_mask).sum().cpu())
            mae_num += float(((pred - br).abs() * beval_mask).sum().cpu())
            mae_den += float(beval_mask.sum().cpu())
    return nll_num / max(nll_den, 1.0), mae_num / max(mae_den, 1.0)

def train_autoencoder(ratio_t: torch.Tensor, m6a_t: torch.Tensor, total_t: torch.Tensor, train_t: torch.Tensor, heldout_t: torch.Tensor, sig_heldout_t: torch.Tensor):
    model = BinomialAutoencoder(ratio_t.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loader = DataLoader(TensorDataset(torch.arange(ratio_t.shape[0], dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)
    rows = []
    best = {'score': math.inf, 'epoch': 0, 'state': None}
    stale = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        batch_losses = []
        for (idx_cpu,) in loader:
            idx = idx_cpu.to(DEVICE)
            br = ratio_t.index_select(0, idx)
            bm6a = m6a_t.index_select(0, idx)
            bt = total_t.index_select(0, idx)
            btrain = train_t.index_select(0, idx)
            pred, _ = model(make_input_features(br, btrain, INPUT_NOISE))
            loss = masked_binomial_nll(pred, bm6a, bt, btrain)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            batch_losses.append(float(loss.detach().cpu()))
        heldout_nll, heldout_mae = evaluate_model(model, train_t, heldout_t, ratio_t, m6a_t, total_t)
        sig_nll, sig_mae = evaluate_model(model, train_t, sig_heldout_t, ratio_t, m6a_t, total_t)
        score = SELECTION_SIGNIFICANT_WEIGHT * sig_nll + (1 - SELECTION_SIGNIFICANT_WEIGHT) * heldout_nll
        eligible = epoch >= MIN_EPOCHS
        row = {'epoch': epoch, 'train_nll': float(np.mean(batch_losses)), 'heldout_nll': heldout_nll, 'heldout_mae': heldout_mae, 'significant_heldout_nll': sig_nll, 'significant_heldout_mae': sig_mae, 'selection_score': score, 'eligible_for_selection': eligible}
        rows.append(row)
        print(row, flush=True)
        if eligible and score < best['score']:
            best = {'score': score, 'epoch': epoch, 'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
            stale = 0
        elif eligible:
            stale += 1
            if stale >= PATIENCE:
                break
    if best['state'] is None:
        best = {'score': rows[-1]['selection_score'], 'epoch': rows[-1]['epoch'], 'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(best['state'])
    model.to(DEVICE)
    return model, pd.DataFrame(rows), int(best['epoch'])


In [ ]:
def fit_randomized_pca_mps(ratio_t: torch.Tensor, mask_t: torch.Tensor, n_components: int = PCA_COMPONENTS, oversample: int = PCA_OVERSAMPLE):
    n_cells, n_sites = ratio_t.shape
    q = min(n_components + oversample, n_cells - 1)
    site_mean_t = site_mean_prediction(ratio_t, mask_t)
    centered_t = (ratio_t - site_mean_t.unsqueeze(0)) * mask_t
    omega = torch.randn((n_sites, q), dtype=torch.float32, device=DEVICE) / math.sqrt(q)
    y = centered_t @ omega
    qmat, _ = torch.linalg.qr(y, mode='reduced')
    components_t = None
    try:
        bmat = qmat.T @ centered_t
        umat, sval, vh = torch.linalg.svd(bmat, full_matrices=False)
        scores_t = qmat @ umat[:, :n_components] * sval[:n_components].unsqueeze(0)
        components_t = vh[:n_components]
        note = 'randomized_pca_svd'
    except Exception as exc:
        scores_t = qmat[:, :n_components]
        note = f'randomized_range_only_no_reconstruction: {type(exc).__name__}: {exc}'
    del centered_t
    return scores_t, site_mean_t, components_t, note

def evaluate_pca_reconstruction(site_mean_t: torch.Tensor, components_t: torch.Tensor | None, train_t: torch.Tensor, eval_t: torch.Tensor, ratio_t: torch.Tensor, m6a_t: torch.Tensor, total_t: torch.Tensor):
    if components_t is None:
        return float('nan'), float('nan')
    nll_num = nll_den = mae_num = mae_den = 0.0
    with torch.inference_mode():
        for start in range(0, ratio_t.shape[0], 32):
            sl = slice(start, min(start + 32, ratio_t.shape[0]))
            centered = (ratio_t[sl] - site_mean_t.unsqueeze(0)) * train_t[sl]
            pred = torch.clamp((centered @ components_t.T) @ components_t + site_mean_t.unsqueeze(0), 0.0, 1.0)
            bt, bm6a, beval = total_t[sl], m6a_t[sl], eval_t[sl]
            p = torch.clamp(pred, 1e-6, 1 - 1e-6)
            effective_total = torch.clamp(bt, max=TOTAL_CAP)
            weights = effective_total / torch.clamp(bt, min=1.0)
            nll = -(bm6a * torch.log(p) + (bt - bm6a) * torch.log1p(-p)) * weights
            nll_num += float((nll * beval).sum().cpu())
            nll_den += float((effective_total * beval).sum().cpu())
            mae_num += float(((pred - ratio_t[sl]).abs() * beval).sum().cpu())
            mae_den += float(beval.sum().cpu())
    return nll_num / max(nll_den, 1.0), mae_num / max(mae_den, 1.0)

def pca_scores_small_mps(x_np: np.ndarray, n_components: int = 2) -> np.ndarray:
    x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    x = x - x.mean(dim=0, keepdim=True)
    u, s, _ = torch.linalg.svd(x, full_matrices=False)
    return (u[:, :n_components] * s[:n_components].unsqueeze(0)).cpu().numpy()

def kmeans_mps(x_np: np.ndarray, k: int, seed: int, n_init: int = 10, max_iter: int = 100):
    x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    n = x.shape[0]
    best_labels = None
    best_centers = None
    best_inertia = math.inf
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    for _ in range(n_init):
        perm = torch.randperm(n, device=DEVICE, generator=gen)
        centers = x[perm[:k]].clone()
        labels = torch.zeros(n, dtype=torch.long, device=DEVICE)
        for _ in range(max_iter):
            dist = ((x[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)
            new_labels = dist.argmin(dim=1)
            if torch.equal(new_labels, labels):
                break
            labels = new_labels
            for j in range(k):
                pts = x[labels == j]
                centers[j] = x[torch.randint(0, n, (1,), device=DEVICE, generator=gen)[0]] if pts.shape[0] == 0 else pts.mean(dim=0)
        inertia = float(((x - centers[labels]) ** 2).sum().cpu())
        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels.detach().cpu().numpy()
            best_centers = centers.detach().cpu().numpy()
    return best_labels, best_centers, best_inertia

def calinski_harabasz(x_np: np.ndarray, labels: np.ndarray, centers: np.ndarray, inertia: float) -> float:
    n = x_np.shape[0]
    k = len(np.unique(labels))
    if k <= 1 or k >= n:
        return float('-inf')
    overall = x_np.mean(axis=0)
    between = 0.0
    for j in range(k):
        count = int((labels == j).sum())
        if count > 0:
            between += count * float(((centers[j] - overall) ** 2).sum())
    return (between / (k - 1)) / (max(inertia, 1e-8) / (n - k))

def choose_k_mps(x_np: np.ndarray, seed: int):
    rows = []
    best = {'k': 2, 'score': float('-inf'), 'labels': None}
    for k in [2, 3, 4, 5, 6]:
        labels, centers, inertia = kmeans_mps(x_np, k, seed + k)
        score = calinski_harabasz(x_np, labels, centers, inertia)
        rows.append({'k': k, 'calinski_harabasz': score, 'inertia': inertia})
        if score > best['score']:
            best = {'k': k, 'score': score, 'labels': labels}
    return best['k'], pd.DataFrame(rows)

def adjusted_rand_index(labels_true, labels_pred) -> float:
    contingency = pd.crosstab(labels_true, labels_pred).to_numpy(dtype=np.int64)
    def comb2(x):
        x = x.astype(np.float64)
        return float((x * (x - 1) / 2).sum())
    sum_cells = comb2(contingency.ravel())
    sum_rows = comb2(contingency.sum(axis=1))
    sum_cols = comb2(contingency.sum(axis=0))
    n = contingency.sum()
    total = n * (n - 1) / 2
    expected = sum_rows * sum_cols / total if total else 0.0
    max_index = 0.5 * (sum_rows + sum_cols)
    return 0.0 if max_index == expected else float((sum_cells - expected) / (max_index - expected))

def neighbor_overlap_mps(reference_np: np.ndarray, query_np: np.ndarray, k: int = NEIGHBOR_K) -> float:
    ref = torch.tensor(reference_np, dtype=torch.float32, device=DEVICE)
    qry = torch.tensor(query_np, dtype=torch.float32, device=DEVICE)
    k = min(k, ref.shape[0] - 1)
    def neigh(x):
        norm = (x * x).sum(dim=1, keepdim=True)
        dist = norm + norm.T - 2 * (x @ x.T)
        dist.fill_diagonal_(float('inf'))
        return torch.topk(dist, k=k, largest=False).indices.cpu().numpy()
    a = neigh(ref)
    b = neigh(qry)
    return float(np.mean([len(set(x).intersection(set(y))) / k for x, y in zip(a, b)]))

def rank_corr(x: np.ndarray, y: np.ndarray) -> float:
    xr = pd.Series(x).rank(method='average').to_numpy(dtype=np.float32)
    yr = pd.Series(y).rank(method='average').to_numpy(dtype=np.float32)
    if xr.std() == 0 or yr.std() == 0:
        return 0.0
    return float(np.corrcoef(xr, yr)[0, 1])


In [ ]:
ratio_cell_ids, site_ids, ratio_np = load_matrix(INPUT_DIR / 'step02_ratio_cells_by_sites.csv.gz')
m6a_cell_ids, _, m6a_np = load_matrix(INPUT_DIR / 'step02_m6a_cells_by_sites.csv.gz')
total_cell_ids, _, total_np = load_matrix(INPUT_DIR / 'step02_total_cells_by_sites.csv.gz')
coverage_cell_ids, _, coverage_np = load_matrix(INPUT_DIR / 'step02_coverage_mask_cells_by_sites.csv.gz')
train_cell_ids, _, train_np = load_matrix(INPUT_DIR / 'step02_train_mask_cells_by_sites.csv.gz')
heldout_cell_ids, _, heldout_np = load_matrix(INPUT_DIR / 'step02_heldout_mask_cells_by_sites.csv.gz')
sig_cell_ids, _, sig_np = load_matrix(INPUT_DIR / 'step02_significance_mask_cells_by_sites.csv.gz')
expr_cell_ids, gene_ids, expression_np = load_matrix(INPUT_DIR / 'step02_expression_anchor_cells_by_genes.csv.gz')
cell_metadata = pd.read_csv(INPUT_DIR / 'step02_cell_metadata.csv.gz')
site_metadata = pd.read_csv(INPUT_DIR / 'step02_site_metadata.csv.gz')
regulator_df = pd.read_csv(INPUT_DIR / 'step02_regulator_expression_by_cell.csv.gz')

for name, ids in [('m6a', m6a_cell_ids), ('total', total_cell_ids), ('coverage', coverage_cell_ids), ('train', train_cell_ids), ('heldout', heldout_cell_ids), ('significance', sig_cell_ids), ('expression', expr_cell_ids)]:
    if not np.array_equal(ratio_cell_ids, ids):
        raise ValueError(f'Cell order mismatch: ratio vs {name}')

ratio_t = to_mps(ratio_np)
m6a_t = to_mps(m6a_np)
total_t = to_mps(total_np)
coverage_t = to_mps(coverage_np)
train_t = to_mps(train_np)
heldout_t = to_mps(heldout_np)
sig_t = to_mps(sig_np)
sig_heldout_t = heldout_t * sig_t
expression_t = to_mps(expression_np)

del ratio_np, m6a_np, total_np, coverage_np, train_np, heldout_np, sig_np
gc.collect()

print({'cells': int(ratio_t.shape[0]), 'sites': int(ratio_t.shape[1]), 'heldout_entries': int(heldout_t.sum().cpu()), 'significant_heldout_entries': int(sig_heldout_t.sum().cpu())})


In [ ]:
run_start = time.time()

site_mean_t = site_mean_prediction(ratio_t, train_t)
site_all_nll, site_all_mae = evaluate_prediction_vector(site_mean_t, heldout_t, ratio_t, m6a_t, total_t)
site_sig_nll, site_sig_mae = evaluate_prediction_vector(site_mean_t, sig_heldout_t, ratio_t, m6a_t, total_t)
print({'site_mean_all_nll': site_all_nll, 'site_mean_significant_nll': site_sig_nll})

print('fitting randomized m6A PCA baseline on MPS')
m6a_pca_scores_t, pca_site_mean_t, pca_components_t, pca_note = fit_randomized_pca_mps(ratio_t, train_t, PCA_COMPONENTS, PCA_OVERSAMPLE)
pca_all_nll, pca_all_mae = evaluate_pca_reconstruction(pca_site_mean_t, pca_components_t, train_t, heldout_t, ratio_t, m6a_t, total_t)
pca_sig_nll, pca_sig_mae = evaluate_pca_reconstruction(pca_site_mean_t, pca_components_t, train_t, sig_heldout_t, ratio_t, m6a_t, total_t)
print({'m6a_pca_note': pca_note, 'm6a_pca_all_nll': pca_all_nll, 'm6a_pca_significant_nll': pca_sig_nll})

print('training autoencoder on MPS')
model, history_df, best_epoch = train_autoencoder(ratio_t, m6a_t, total_t, train_t, heldout_t, sig_heldout_t)
ae_all_nll, ae_all_mae = evaluate_model(model, train_t, heldout_t, ratio_t, m6a_t, total_t)
ae_sig_nll, ae_sig_mae = evaluate_model(model, train_t, sig_heldout_t, ratio_t, m6a_t, total_t)

method_metrics = pd.DataFrame([
    {'method': 'site_mean', 'heldout_nll': site_all_nll, 'heldout_mae': site_all_mae, 'significant_heldout_nll': site_sig_nll, 'significant_heldout_mae': site_sig_mae},
    {'method': 'm6a_pca', 'heldout_nll': pca_all_nll, 'heldout_mae': pca_all_mae, 'significant_heldout_nll': pca_sig_nll, 'significant_heldout_mae': pca_sig_mae},
    {'method': 'dca_like_ae', 'heldout_nll': ae_all_nll, 'heldout_mae': ae_all_mae, 'significant_heldout_nll': ae_sig_nll, 'significant_heldout_mae': ae_sig_mae},
])
display(method_metrics)


In [ ]:
def extract_latent(model: nn.Module, input_mask_t: torch.Tensor):
    model.eval()
    zs = []
    with torch.inference_mode():
        for start in range(0, ratio_t.shape[0], BATCH_SIZE):
            sl = slice(start, min(start + BATCH_SIZE, ratio_t.shape[0]))
            _, z = model(make_input_features(ratio_t[sl], input_mask_t[sl], 0.0))
            zs.append(z.cpu().numpy())
    z_np = np.vstack(zs).astype(np.float32)
    return (z_np - z_np.mean(axis=0, keepdims=True)) / np.maximum(z_np.std(axis=0, keepdims=True), 1e-6)

ae_latent = extract_latent(model, coverage_t)
ae_2d = pca_scores_small_mps(ae_latent, 2)
m6a_pca_scores = m6a_pca_scores_t.cpu().numpy().astype(np.float32)
m6a_pca_2d = pca_scores_small_mps(m6a_pca_scores, 2) if m6a_pca_scores.shape[1] > 2 else m6a_pca_scores[:, :2]
expression_pcs = pca_scores_small_mps(expression_np.astype(np.float32), min(10, expression_np.shape[1], expression_np.shape[0] - 1))

cluster_k, cluster_selection_df = choose_k_mps(expression_pcs, SEED)
expr_cluster, _, _ = kmeans_mps(expression_pcs, cluster_k, SEED + 100)
m6a_cluster, _, _ = kmeans_mps(m6a_pca_scores, cluster_k, SEED + 200)
ae_cluster, _, _ = kmeans_mps(ae_latent, cluster_k, SEED + 300)

expr_vs_ae_ari = adjusted_rand_index(expr_cluster, ae_cluster)
expr_vs_m6a_pca_ari = adjusted_rand_index(expr_cluster, m6a_cluster)
expr_vs_ae_neighbor_overlap = neighbor_overlap_mps(expression_pcs, ae_latent, NEIGHBOR_K)
expr_vs_m6a_pca_neighbor_overlap = neighbor_overlap_mps(expression_pcs, m6a_pca_scores, NEIGHBOR_K)

cluster_metrics = pd.DataFrame([
    {'comparison': 'expression_vs_ae', 'ari': expr_vs_ae_ari, 'neighbor_overlap': expr_vs_ae_neighbor_overlap},
    {'comparison': 'expression_vs_m6a_pca', 'ari': expr_vs_m6a_pca_ari, 'neighbor_overlap': expr_vs_m6a_pca_neighbor_overlap},
])

reg_rows = []
for regulator in regulator_df.columns[1:]:
    values = regulator_df[regulator].to_numpy(dtype=np.float32)
    for dim_idx in range(ae_latent.shape[1]):
        corr = rank_corr(ae_latent[:, dim_idx], values)
        reg_rows.append({'pair': f'{regulator} vs latent_{dim_idx + 1}', 'regulator': regulator, 'latent_dim': dim_idx + 1, 'spearman_like_corr': corr, 'abs_corr': abs(corr)})
regulator_corr_df = pd.DataFrame(reg_rows).sort_values('abs_corr', ascending=False).reset_index(drop=True)

cell_embeddings = cell_metadata.copy()
cell_embeddings['expression_pc1'] = expression_pcs[:, 0]
cell_embeddings['expression_pc2'] = expression_pcs[:, 1]
cell_embeddings['expression_cluster'] = expr_cluster.astype(str)
cell_embeddings['m6a_pca1'] = m6a_pca_2d[:, 0]
cell_embeddings['m6a_pca2'] = m6a_pca_2d[:, 1]
cell_embeddings['m6a_pca_cluster'] = m6a_cluster.astype(str)
cell_embeddings['ae_pc1'] = ae_2d[:, 0]
cell_embeddings['ae_pc2'] = ae_2d[:, 1]
cell_embeddings['ae_cluster'] = ae_cluster.astype(str)

display(cluster_metrics)
display(regulator_corr_df.head(12))


In [ ]:
def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.show()

history_df.to_csv(OUTPUT_DIR / 'step03_training_history.csv', index=False)
method_metrics.to_csv(OUTPUT_DIR / 'step03_method_metrics.csv', index=False)
cluster_selection_df.to_csv(OUTPUT_DIR / 'step03_cluster_selection_metrics.csv', index=False)
cluster_metrics.to_csv(OUTPUT_DIR / 'step03_cluster_comparison_metrics.csv', index=False)
regulator_corr_df.to_csv(OUTPUT_DIR / 'step03_regulator_correlations.csv', index=False)
cell_embeddings.to_csv(OUTPUT_DIR / 'step03_cell_embeddings.csv', index=False)
site_metadata.to_csv(OUTPUT_DIR / 'step03_site_metadata_snapshot.csv', index=False)
torch.save(model.state_dict(), OUTPUT_DIR / 'step03_final_model_mps.pt')
np.savez_compressed(OUTPUT_DIR / 'step03_embeddings.npz', ae_latent=ae_latent, m6a_pca_scores=m6a_pca_scores, expression_pcs=expression_pcs, cell_ids=ratio_cell_ids.astype(str), site_ids=site_ids.astype(str))

plt.figure(figsize=(8, 5))
plt.plot(history_df['epoch'], history_df['train_nll'], label='train_nll')
plt.plot(history_df['epoch'], history_df['heldout_nll'], label='heldout_nll')
plt.plot(history_df['epoch'], history_df['significant_heldout_nll'], label='significant_heldout_nll')
plt.xlabel('epoch')
plt.ylabel('capped binomial NLL')
plt.legend()
plt.grid(alpha=0.3)
savefig(OUTPUT_DIR / 'plot_training_nll_curves.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=method_metrics, x='method', y='significant_heldout_nll', ax=axes[0])
axes[0].set_title('Significant heldout NLL')
sns.barplot(data=method_metrics, x='method', y='heldout_nll', ax=axes[1])
axes[1].set_title('All heldout NLL')
savefig(OUTPUT_DIR / 'plot_method_nll_comparison.png')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, x, y, title in [(axes[0], 'expression_pc1', 'expression_pc2', 'Expression PCA'), (axes[1], 'm6a_pca1', 'm6a_pca2', 'm6A randomized PCA'), (axes[2], 'ae_pc1', 'ae_pc2', 'AE latent PCA')]:
    sns.scatterplot(data=cell_embeddings, x=x, y=y, hue='ae_cluster', palette='tab10', s=24, linewidth=0, alpha=0.85, ax=ax)
    ax.set_title(title)
    if ax.legend_ is not None:
        ax.legend_.remove()
handles, labels = axes[2].get_legend_handles_labels()
fig.legend(handles, labels, title='AE cluster', loc='center right')
plt.tight_layout(rect=(0, 0, 0.92, 1))
plt.savefig(OUTPUT_DIR / 'plot_embedding_panels_by_ae_cluster.png', dpi=220)
plt.show()

cluster_counts = cell_embeddings['ae_cluster'].value_counts().sort_index().reset_index()
cluster_counts.columns = ['ae_cluster', 'n_cells']
plt.figure(figsize=(6, 4))
sns.barplot(data=cluster_counts, x='ae_cluster', y='n_cells')
plt.title('AE cluster counts')
savefig(OUTPUT_DIR / 'plot_ae_cluster_counts.png')

plt.figure(figsize=(7, 4))
sns.barplot(data=cluster_metrics, x='comparison', y='neighbor_overlap')
plt.title(f'Neighbor overlap with expression PCA, k={NEIGHBOR_K}')
plt.xticks(rotation=20, ha='right')
savefig(OUTPUT_DIR / 'plot_neighbor_overlap.png')

if not regulator_corr_df.empty:
    top = regulator_corr_df.head(15).copy()
    plt.figure(figsize=(8, 5))
    sns.barplot(data=top, x='spearman_like_corr', y='pair', hue=(top['spearman_like_corr'] > 0), dodge=False, legend=False)
    plt.axvline(0, color='black', linewidth=1)
    plt.title('Top AE latent vs m6A regulator correlations')
    savefig(OUTPUT_DIR / 'plot_top_regulator_correlations.png')

    heat = regulator_corr_df.pivot(index='regulator', columns='latent_dim', values='spearman_like_corr')
    plt.figure(figsize=(10, 6))
    sns.heatmap(heat, cmap='vlag', center=0)
    plt.title('Regulator expression vs AE latent dimensions')
    savefig(OUTPUT_DIR / 'plot_regulator_correlation_heatmap.png')


In [ ]:
def sample_significant_predictions(model: nn.Module, max_points: int = PREDICTION_SCATTER_MAX_POINTS):
    obs_all = []
    pred_all = []
    with torch.inference_mode():
        for start in range(0, ratio_t.shape[0], 16):
            sl = slice(start, min(start + 16, ratio_t.shape[0]))
            pred, _ = model(make_input_features(ratio_t[sl], train_t[sl], 0.0))
            mask = sig_heldout_t[sl] > 0
            if int(mask.sum().cpu()) > 0:
                obs_all.append(ratio_t[sl][mask].detach().cpu().numpy())
                pred_all.append(pred[mask].detach().cpu().numpy())
    if not obs_all:
        return pd.DataFrame({'observed_ratio': [], 'predicted_ratio': []})
    obs = np.concatenate(obs_all)
    pred = np.concatenate(pred_all)
    if len(obs) > max_points:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(obs), size=max_points, replace=False)
        obs, pred = obs[idx], pred[idx]
    return pd.DataFrame({'observed_ratio': obs, 'predicted_ratio': pred})

scatter_df = sample_significant_predictions(model)
scatter_df.to_csv(OUTPUT_DIR / 'step03_significant_heldout_prediction_sample.csv', index=False)
if len(scatter_df) > 0:
    plt.figure(figsize=(5.5, 5))
    sns.scatterplot(data=scatter_df, x='observed_ratio', y='predicted_ratio', s=8, alpha=0.25, linewidth=0)
    plt.plot([0, 1], [0, 1], color='black', linewidth=1)
    plt.title('Significant heldout: observed vs AE predicted ratio')
    savefig(OUTPUT_DIR / 'plot_significant_heldout_observed_vs_predicted.png')

score = {
    'run_id': 'no_site_filter_all135k_notebook_mps_pca_cluster',
    'input_dir': str(INPUT_DIR),
    'output_dir': str(OUTPUT_DIR),
    'device': str(DEVICE),
    'cells': int(ratio_t.shape[0]),
    'sites': int(ratio_t.shape[1]),
    'heldout_entries': int(heldout_t.sum().cpu()),
    'significant_heldout_entries': int(sig_heldout_t.sum().cpu()),
    'best_epoch': int(best_epoch),
    'selected_cluster_k': int(cluster_k),
    'site_mean_significant_nll': float(site_sig_nll),
    'm6a_pca_significant_nll': float(pca_sig_nll),
    'dca_like_significant_nll': float(ae_sig_nll),
    'significant_delta_nll_vs_site_mean': float(site_sig_nll - ae_sig_nll),
    'site_mean_all_nll': float(site_all_nll),
    'm6a_pca_all_nll': float(pca_all_nll),
    'dca_like_all_nll': float(ae_all_nll),
    'all_delta_nll_vs_site_mean': float(site_all_nll - ae_all_nll),
    'expression_vs_ae_ari': float(expr_vs_ae_ari),
    'expression_vs_m6a_pca_ari': float(expr_vs_m6a_pca_ari),
    'expression_vs_ae_neighbor_overlap': float(expr_vs_ae_neighbor_overlap),
    'expression_vs_m6a_pca_neighbor_overlap': float(expr_vs_m6a_pca_neighbor_overlap),
    'ae_cluster_counts': cell_embeddings['ae_cluster'].value_counts().sort_index().to_dict(),
    'top_regulator_pair': str(regulator_corr_df.iloc[0]['pair']) if not regulator_corr_df.empty else 'NA',
    'top_regulator_corr': float(regulator_corr_df.iloc[0]['spearman_like_corr']) if not regulator_corr_df.empty else float('nan'),
    'pca_note': pca_note,
    'elapsed_minutes': round((time.time() - run_start) / 60, 3),
    'max_epochs': MAX_EPOCHS,
    'min_epochs': MIN_EPOCHS,
    'patience': PATIENCE,
    'latent_dim': LATENT_DIM,
    'hidden_dim_1': HIDDEN_DIM_1,
    'hidden_dim_2': HIDDEN_DIM_2,
    'dropout': DROPOUT,
    'input_noise': INPUT_NOISE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'total_cap': TOTAL_CAP,
}
(OUTPUT_DIR / 'step03_score.json').write_text(json.dumps(score, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'step03_summary.txt').write_text('\n'.join([f'{k}={v}' for k, v in score.items()]) + '\n', encoding='utf-8')
print(json.dumps(score, indent=2))
